# 03 — Final Results Verification

This notebook verifies the final experimental result set before statistical analysis. It checks experiment completeness, seeds, provenance, metric consistency, relative-drop calculations, RecBole dataset integrity, sparsity construction, raw-to-processed traceability, and project tests.

In [1]:
from pathlib import Path
import hashlib
import json
import re
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# Locate repository root
# ---------------------------------------------------------------------

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "results").exists()
            and (candidate / "scripts").exists()
            and (candidate / "configs").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root. "
        "Run this notebook from somewhere inside the project repository."
    )


ROOT = find_repo_root()
VERIFICATION_DIR = ROOT / "results" / "verification"
VERIFICATION_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root : {ROOT}")
print(f"Verification dir: {VERIFICATION_DIR}")


# ---------------------------------------------------------------------
# Verification recorder
# ---------------------------------------------------------------------

CHECKS = []


def add_check(section, name, condition, details="", severity="FAIL"):
    """
    severity:
        FAIL -> genuine verification failure
        WARN -> something requiring review, but not automatically invalid
    """
    passed = bool(condition)
    status = "PASS" if passed else severity

    CHECKS.append({
        "section": section,
        "check": name,
        "status": status,
        "details": str(details),
    })


def show_checks(section_prefix=None):
    df = pd.DataFrame(CHECKS)

    if section_prefix is not None and not df.empty:
        df = df[df["section"].str.startswith(section_prefix)]

    display(df.reset_index(drop=True))


# ---------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


def normalise_recbole_columns(df):
    """Convert e.g. user_id:token -> user_id."""
    return df.rename(columns={c: c.split(":")[0] for c in df.columns})


def read_recbole_table(path):
    return normalise_recbole_columns(
        pd.read_csv(path, sep="\t", low_memory=False)
    )


def find_single_file(directory, suffix):
    files = sorted(Path(directory).glob(f"*{suffix}"))

    if len(files) != 1:
        return None

    return files[0]


def row_hashes(df, columns):
    """
    Stable row hashes used for exact set/multiset comparisons.
    """
    temp = df[columns].copy()

    for col in columns:
        temp[col] = temp[col].astype(str)

    return np.sort(
        pd.util.hash_pandas_object(
            temp,
            index=False
        ).to_numpy(dtype=np.uint64)
    )


def same_rows(df1, df2, columns):
    a = row_hashes(df1, columns)
    b = row_hashes(df2, columns)
    return len(a) == len(b) and np.array_equal(a, b)


def subset_rows(smaller, larger, columns):
    a = np.unique(row_hashes(smaller, columns))
    b = np.unique(row_hashes(larger, columns))

    return len(np.setdiff1d(a, b, assume_unique=True)) == 0


def split_leave_one_out(df):
    """
    Independently reconstruct RecBole temporal leave-one-out:
        final interaction      -> test
        second-last interaction -> validation
        everything before      -> train
    """
    ordered = df.sort_values(
        ["user_id", "sequence_order"],
        kind="mergesort"
    ).copy()

    ordered["_position"] = ordered.groupby(
        "user_id", sort=False
    ).cumcount()

    ordered["_history_size"] = ordered.groupby(
        "user_id", sort=False
    )["user_id"].transform("size")

    ordered["_from_end"] = (
        ordered["_history_size"]
        - ordered["_position"]
        - 1
    )

    train = ordered[ordered["_from_end"] >= 2].copy()
    validation = ordered[ordered["_from_end"] == 1].copy()
    test = ordered[ordered["_from_end"] == 0].copy()

    return train, validation, test


# ---------------------------------------------------------------------
# Experiment definition
# ---------------------------------------------------------------------

MODELS = [
    "Pop",
    "ItemKNN",
    "BPR",
    "EASE",
    "NeuMF",
    "MultiVAE",
    "GRU4Rec",
    "SASRec",
    "BERT4Rec",
    "LightGCN",
]

DETERMINISTIC_MODELS = {
    "Pop",
    "ItemKNN",
    "EASE",
}

STOCHASTIC_MODELS = set(MODELS) - DETERMINISTIC_MODELS

EXPECTED_SEEDS = {
    model: ([2025] if model in DETERMINISTIC_MODELS else [2025, 2026, 2027])
    for model in MODELS
}

EXPECTED_CONDITIONS = [
    ("baseline", 100),
    ("global", 50),
    ("global", 25),
    ("global", 10),
    ("recent", 50),
    ("recent", 25),
    ("recent", 10),
    ("early", 50),
    ("early", 25),
    ("early", 10),
]

EXPECTED_SCENARIO_METHOD = {
    "baseline": None,
    "global": "random_per_user",
    "recent": "recent_history",
    "early": "early_profile",
}

EXPECTED_DATASET_STATS = {
    "movielens": {
        "users": 6034,
        "items": 3125,
        "interactions": 574376,
        "train": 562308,
        "validation": 6034,
        "test": 6034,
        "max_item_list_length": 200,
    },
    "amazon": {
        "users": 63413,
        "items": 19022,
        "interactions": 533550,
        "train": 406724,
        "validation": 63413,
        "test": 63413,
        "max_item_list_length": 50,
    },
}

Repository root : C:\Research Project\recommender-sparsity-honours
Verification dir: C:\Research Project\recommender-sparsity-honours\results\verification


## 1.2 Load final results

Load the final MovieLens and Amazon result tables and confirm that both datasets are present.

In [2]:
result_files = sorted(
    (ROOT / "results" / "processed").glob("*/all_final_results.csv")
)

if not result_files:
    result_files = sorted(
        (ROOT / "results" / "processed").rglob("all_final_results.csv")
    )

if not result_files:
    raise FileNotFoundError(
        "No all_final_results.csv files were found under results/processed/."
    )


frames = []

for path in result_files:
    df = pd.read_csv(path)

    if "dataset" not in df.columns:
        raise ValueError(f"'dataset' column missing from {path}")

    df["_source_file"] = str(path.relative_to(ROOT))
    frames.append(df)

results = pd.concat(frames, ignore_index=True)

results["model_seed"] = pd.to_numeric(
    results["model_seed"], errors="coerce"
).astype("Int64")

results["retention_level"] = pd.to_numeric(
    results["retention_level"], errors="coerce"
).astype("Int64")


print(f"Files loaded : {len(result_files)}")
print(f"Total rows   : {len(results)}")
print()

display(
    results.groupby("dataset")
    .agg(
        rows=("model", "size"),
        models=("model", "nunique"),
        scenarios=("scenario", "nunique"),
    )
)

add_check(
    "1.2",
    "Both expected datasets loaded",
    set(results["dataset"].unique()) == {"movielens", "amazon"},
    f"Found: {sorted(results['dataset'].unique())}",
)

add_check(
    "1.2",
    "Result tables contain 480 rows in total",
    len(results) == 480,
    f"Rows found: {len(results)}",
)

show_checks("1.2")

Files loaded : 2
Total rows   : 480



,rows,models,scenarios
dataset,,,
amazon,240,10,4
movielens,240,10,4


,section,check,status,details
0,1.2,Both expected datasets loaded,PASS,"Found: ['amazon', 'movielens']"
1,1.2,Result tables contain 480 rows in total,PASS,Rows found: 480


# 2. Experiment Completeness

## 2.1 Expected experiment grid

Verify that every expected dataset × condition × model × seed combination exists exactly once.

In [3]:
# ---------------------------------------------------------------------
# Construct expected experiment grid
# ---------------------------------------------------------------------

expected_rows = []

for dataset in ["movielens", "amazon"]:
    for scenario, retention in EXPECTED_CONDITIONS:
        for model in MODELS:
            for seed in EXPECTED_SEEDS[model]:
                expected_rows.append({
                    "dataset": dataset,
                    "scenario": scenario,
                    "retention_level": retention,
                    "model": model,
                    "model_seed": seed,
                })

expected_grid = pd.DataFrame(expected_rows)
expected_grid["model_seed"] = expected_grid["model_seed"].astype("Int64")
expected_grid["retention_level"] = expected_grid["retention_level"].astype("Int64")

KEY_COLS = [
    "dataset",
    "scenario",
    "retention_level",
    "model",
    "model_seed",
]

actual_grid = results[KEY_COLS].copy()

duplicates = actual_grid[
    actual_grid.duplicated(KEY_COLS, keep=False)
].sort_values(KEY_COLS)

comparison = expected_grid.merge(
    actual_grid.drop_duplicates(),
    on=KEY_COLS,
    how="outer",
    indicator=True,
)

missing = comparison[comparison["_merge"] == "left_only"]
unexpected = comparison[comparison["_merge"] == "right_only"]


# ---------------------------------------------------------------------
# Checks
# ---------------------------------------------------------------------

for dataset in ["movielens", "amazon"]:
    n = len(results[results["dataset"] == dataset])

    add_check(
        "2.1",
        f"{dataset}: exactly 240 final rows",
        n == 240,
        f"Rows found: {n}",
    )

    condition_counts = (
        results[results["dataset"] == dataset]
        .groupby(["scenario", "retention_level"])
        .size()
    )

    add_check(
        "2.1",
        f"{dataset}: 24 rows per experimental condition",
        len(condition_counts) == 10
        and (condition_counts == 24).all(),
        condition_counts.to_dict(),
    )


add_check(
    "2.1",
    "No duplicate experiment keys",
    duplicates.empty,
    f"Duplicate rows: {len(duplicates)}",
)

add_check(
    "2.1",
    "No expected experiments are missing",
    missing.empty,
    f"Missing combinations: {len(missing)}",
)

add_check(
    "2.1",
    "No unexpected experiment combinations",
    unexpected.empty,
    f"Unexpected combinations: {len(unexpected)}",
)

add_check(
    "2.1",
    "Exactly 10 expected models",
    set(results["model"].unique()) == set(MODELS),
    sorted(results["model"].unique()),
)


# ---------------------------------------------------------------------
# Display experiment inventory
# ---------------------------------------------------------------------

inventory = (
    results.groupby(
        ["dataset", "scenario", "retention_level", "model"],
        as_index=False
    )
    .agg(
        runs=("model_seed", "size"),
        seeds=("model_seed", lambda x: ",".join(map(str, sorted(x.dropna().astype(int))))),
    )
)

display(inventory)

if not missing.empty:
    print("MISSING EXPERIMENTS")
    display(missing)

if not unexpected.empty:
    print("UNEXPECTED EXPERIMENTS")
    display(unexpected)

if not duplicates.empty:
    print("DUPLICATE EXPERIMENT KEYS")
    display(
        results.loc[duplicates.index, KEY_COLS + ["timestamp"]]
        .sort_values(KEY_COLS)
    )

show_checks("2.1")

,dataset,scenario,retention_level,model,runs,seeds
0,amazon,baseline,100,BERT4Rec,3,"2025,2026,2027"
1,amazon,baseline,100,BPR,3,"2025,2026,2027"
2,amazon,baseline,100,EASE,1,2025
3,amazon,baseline,100,GRU4Rec,3,"2025,2026,2027"
4,amazon,baseline,100,ItemKNN,1,2025
...,...,...,...,...,...,...
195,movielens,recent,50,LightGCN,3,"2025,2026,2027"
196,movielens,recent,50,MultiVAE,3,"2025,2026,2027"
197,movielens,recent,50,NeuMF,3,"2025,2026,2027"
198,movielens,recent,50,Pop,1,2025


,section,check,status,details
0,2.1,movielens: exactly 240 final rows,PASS,Rows found: 240
1,2.1,movielens: 24 rows per experimental condition,PASS,"{('baseline', 100): 24, ('early', 10): 24, ('e..."
2,2.1,amazon: exactly 240 final rows,PASS,Rows found: 240
3,2.1,amazon: 24 rows per experimental condition,PASS,"{('baseline', 100): 24, ('early', 10): 24, ('e..."
4,2.1,No duplicate experiment keys,PASS,Duplicate rows: 0
5,2.1,No expected experiments are missing,PASS,Missing combinations: 0
6,2.1,No unexpected experiment combinations,PASS,Unexpected combinations: 0
7,2.1,Exactly 10 expected models,PASS,"['BERT4Rec', 'BPR', 'EASE', 'GRU4Rec', 'ItemKN..."


## 2.2 Scenario and retention metadata

Verify the scenario labels, implementation names, nominal retention values, actual retention values, and sparsity seed.

In [4]:
# Experiment type should match the scenario label.
add_check(
    "2.2",
    "experiment_type matches scenario",
    (results["experiment_type"] == results["scenario"]).all(),
)

# Requested retention should equal the named retention level.
add_check(
    "2.2",
    "Requested retention equals retention_level",
    np.allclose(
        results["requested_retention_percent"].astype(float),
        results["retention_level"].astype(float),
    ),
)

# Fraction and percentage must represent the same quantity.
add_check(
    "2.2",
    "Actual retention fraction and percentage agree",
    np.allclose(
        results["actual_training_retention_fraction"] * 100,
        results["actual_training_retention_percent"],
        atol=1e-10,
    ),
)

# Scenario method names.
method_ok = np.ones(len(results), dtype=bool)

for scenario, expected_method in EXPECTED_SCENARIO_METHOD.items():
    mask = results["scenario"] == scenario

    if expected_method is None:
        method_ok[mask] = results.loc[
            mask, "scenario_method"
        ].isna().to_numpy()
    else:
        method_ok[mask] = (
            results.loc[mask, "scenario_method"] == expected_method
        ).to_numpy()

add_check(
    "2.2",
    "Scenario implementation names are correct",
    method_ok.all(),
)

# Random/global sparsity uses seed 2025.
global_mask = results["scenario"] == "global"
nonglobal_mask = ~global_mask

add_check(
    "2.2",
    "Global/random sparsity seed is 2025",
    (
        pd.to_numeric(
            results.loc[global_mask, "sparsity_seed"],
            errors="coerce"
        ) == 2025
    ).all(),
)

add_check(
    "2.2",
    "Non-random scenarios do not record a sparsity seed",
    results.loc[nonglobal_mask, "sparsity_seed"].isna().all(),
)

# Baseline must be exactly 100%.
baseline = results[results["scenario"] == "baseline"]

add_check(
    "2.2",
    "Baseline actual training retention is 100%",
    np.allclose(
        baseline["actual_training_retention_percent"],
        100.0,
    ),
)

# Same nominal retention must produce the same aggregate number of retained
# interactions for recent/global/early, since the per-user retention counts
# are identical and only the selected interactions differ.
retention_consistency = (
    results[results["scenario"] != "baseline"]
    .groupby(["dataset", "retention_level"])
    ["actual_training_retention_percent"]
    .agg(["min", "max"])
)

add_check(
    "2.2",
    "Actual aggregate retention is identical across sparsity scenarios",
    (
        retention_consistency["max"]
        - retention_consistency["min"]
        < 1e-10
    ).all(),
)

condition_summary = (
    results[
        [
            "dataset",
            "scenario",
            "scenario_method",
            "retention_level",
            "requested_retention_percent",
            "actual_training_retention_percent",
            "sparsity_seed",
        ]
    ]
    .drop_duplicates()
    .sort_values(["dataset", "scenario", "retention_level"])
)

display(condition_summary)

show_checks("2.2")

,dataset,scenario,scenario_method,retention_level,requested_retention_percent,actual_training_retention_percent,sparsity_seed
0,amazon,baseline,NaN,100,100,100.000000,NaN
216,amazon,early,early_profile,10,10,18.380032,NaN
192,amazon,early,early_profile,25,25,29.766623,NaN
168,amazon,early,early_profile,50,50,54.711549,NaN
72,amazon,global,random_per_user,10,10,18.380032,2025.0
48,amazon,global,random_per_user,25,25,29.766623,2025.0
24,amazon,global,random_per_user,50,50,54.711549,2025.0
144,amazon,recent,recent_history,10,10,18.380032,NaN
120,amazon,recent,recent_history,25,25,29.766623,NaN
96,amazon,recent,recent_history,50,50,54.711549,NaN


,section,check,status,details
0,2.2,experiment_type matches scenario,PASS,
1,2.2,Requested retention equals retention_level,PASS,
2,2.2,Actual retention fraction and percentage agree,PASS,
3,2.2,Scenario implementation names are correct,PASS,
4,2.2,Global/random sparsity seed is 2025,PASS,
5,2.2,Non-random scenarios do not record a sparsity ...,PASS,
6,2.2,Baseline actual training retention is 100%,PASS,
7,2.2,Actual aggregate retention is identical across...,PASS,


# 3. Provenance and Configuration

## 3.1 Experimental provenance

Verify that the final experiments use frozen model configurations, fixed item catalogues, consistent dataset files, and the intended RecBole evaluation configuration.

In [5]:
# ---------------------------------------------------------------------
# Basic provenance
# ---------------------------------------------------------------------

add_check(
    "3.1",
    "All rows are marked final",
    (results["run_type"] == "final").all(),
    results["run_type"].value_counts().to_dict(),
)

add_check(
    "3.1",
    "Validation metric is NDCG@10",
    (results["valid_metric"] == "NDCG@10").all(),
    results["valid_metric"].value_counts().to_dict(),
)

add_check(
    "3.1",
    "Temporal field is sequence_order",
    (results["time_field"] == "sequence_order").all(),
    results["time_field"].value_counts().to_dict(),
)

add_check(
    "3.1",
    "RecBole version is 1.2.1",
    (results["recbole_version"].astype(str) == "1.2.1").all(),
    sorted(results["recbole_version"].astype(str).unique()),
)


# ---------------------------------------------------------------------
# Frozen model configs
# ---------------------------------------------------------------------

config_counts = (
    results.groupby(["dataset", "model"])
    ["config_sha256"]
    .nunique()
)

add_check(
    "3.1",
    "Each dataset/model uses one frozen config hash",
    (config_counts == 1).all(),
    f"Maximum hashes for any dataset/model: {config_counts.max()}",
)


# ---------------------------------------------------------------------
# Fixed item catalogue
# ---------------------------------------------------------------------

item_hash_counts = (
    results.groupby("dataset")
    ["dataset_item_sha256"]
    .nunique()
)

add_check(
    "3.1",
    "One fixed item catalogue per dataset",
    (item_hash_counts == 1).all(),
    item_hash_counts.to_dict(),
)


# ---------------------------------------------------------------------
# One .inter dataset per experimental condition
# ---------------------------------------------------------------------

inter_condition_counts = (
    results.groupby(["dataset", "scenario", "retention_level"])
    ["dataset_inter_sha256"]
    .nunique()
)

add_check(
    "3.1",
    "One interaction-file hash per experimental condition",
    (inter_condition_counts == 1).all(),
)

for dataset in ["movielens", "amazon"]:
    hashes = (
        results[results["dataset"] == dataset]
        .groupby(["scenario", "retention_level"])
        ["dataset_inter_sha256"]
        .first()
    )

    add_check(
        "3.1",
        f"{dataset}: all 10 conditions have distinct .inter hashes",
        hashes.nunique() == 10,
        f"Unique hashes: {hashes.nunique()}",
    )


# ---------------------------------------------------------------------
# BERT4Rec patch
# ---------------------------------------------------------------------

patch_value = (
    results["bert4rec_patch_active"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

patch_expected = results["model"] == "BERT4Rec"

add_check(
    "3.1",
    "BERT4Rec patch active only for BERT4Rec",
    (patch_value == patch_expected).all(),
)


# ---------------------------------------------------------------------
# Sequence-length settings
# ---------------------------------------------------------------------

for dataset, expected in EXPECTED_DATASET_STATS.items():
    values = (
        results.loc[
            results["dataset"] == dataset,
            "max_item_list_length"
        ]
        .dropna()
        .unique()
    )

    add_check(
        "3.1",
        f"{dataset}: MAX_ITEM_LIST_LENGTH = {expected['max_item_list_length']}",
        len(values) == 1
        and float(values[0]) == expected["max_item_list_length"],
        f"Observed: {values.tolist()}",
    )


# ---------------------------------------------------------------------
# Git/software consistency
# ---------------------------------------------------------------------

for column in [
    "git_commit",
    "torch_version",
    "python_version",
    "recbole_version",
]:
    add_check(
        "3.1",
        f"Single {column} across final result set",
        results[column].nunique(dropna=False) == 1,
        results[column].drop_duplicates().tolist(),
    )


# ---------------------------------------------------------------------
# Runtime/checkpoint metadata
# ---------------------------------------------------------------------

add_check(
    "3.1",
    "Checkpoint paths are recorded",
    results["checkpoint_path"].notna().all()
    and results["checkpoint_path"].astype(str).str.strip().ne("").all(),
)

for column in [
    "training_time_seconds",
    "evaluation_time_seconds",
    "total_time_seconds",
]:
    add_check(
        "3.1",
        f"{column} is positive",
        (
            pd.to_numeric(results[column], errors="coerce") > 0
        ).all(),
    )

add_check(
    "3.1",
    "Total runtime equals training + evaluation runtime",
    np.allclose(
        results["total_time_seconds"],
        results["training_time_seconds"]
        + results["evaluation_time_seconds"],
        atol=2e-6,
    ),
)


# ---------------------------------------------------------------------
# Display provenance summary
# ---------------------------------------------------------------------

provenance_summary = (
    results.groupby("dataset", as_index=False)
    .agg(
        rows=("model", "size"),
        git_commits=("git_commit", "nunique"),
        item_catalogues=("dataset_item_sha256", "nunique"),
        inter_datasets=("dataset_inter_sha256", "nunique"),
        recbole_versions=("recbole_version", "nunique"),
        torch_versions=("torch_version", "nunique"),
        python_versions=("python_version", "nunique"),
    )
)

display(provenance_summary)

show_checks("3.1")

,dataset,rows,git_commits,item_catalogues,inter_datasets,recbole_versions,torch_versions,python_versions
0,amazon,240,1,1,10,1,1,1
1,movielens,240,1,1,10,1,1,1


,section,check,status,details
0,3.1,All rows are marked final,PASS,{'final': 480}
1,3.1,Validation metric is NDCG@10,PASS,{'NDCG@10': 480}
2,3.1,Temporal field is sequence_order,PASS,{'sequence_order': 480}
3,3.1,RecBole version is 1.2.1,PASS,['1.2.1']
4,3.1,Each dataset/model uses one frozen config hash,PASS,Maximum hashes for any dataset/model: 1
5,3.1,One fixed item catalogue per dataset,PASS,"{'amazon': 1, 'movielens': 1}"
6,3.1,One interaction-file hash per experimental con...,PASS,
7,3.1,movielens: all 10 conditions have distinct .in...,PASS,Unique hashes: 10
8,3.1,amazon: all 10 conditions have distinct .inter...,PASS,Unique hashes: 10
9,3.1,BERT4Rec patch active only for BERT4Rec,PASS,


## 3.2 Git state and seed initialisation

Compare the current repository with the commit recorded by the experiments and inspect the final training runner for explicit RecBole seed initialisation.

In [6]:
def run_command(command):
    result = subprocess.run(
        command,
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()


# ---------------------------------------------------------------------
# Git commit
# ---------------------------------------------------------------------

code, current_head, error = run_command(
    ["git", "rev-parse", "HEAD"]
)

if code == 0:
    recorded_commits = results["git_commit"].dropna().unique()

    print(f"Current repository HEAD : {current_head}")
    print(f"Recorded result commit  : {recorded_commits.tolist()}")

    add_check(
        "3.2",
        "Recorded result commit is unique",
        len(recorded_commits) == 1,
        recorded_commits.tolist(),
    )

    if len(recorded_commits) == 1:
        add_check(
            "3.2",
            "Current HEAD equals experimental result commit",
            current_head == recorded_commits[0],
            (
                f"Current={current_head}, "
                f"results={recorded_commits[0]}. "
                "A later analysis/documentation commit is acceptable, "
                "but experimental provenance should remain recorded."
            ),
            severity="WARN",
        )
else:
    add_check(
        "3.2",
        "Git repository readable",
        False,
        error,
        severity="WARN",
    )


# ---------------------------------------------------------------------
# Working tree
# ---------------------------------------------------------------------

code, git_status, error = run_command(
    ["git", "status", "--porcelain"]
)

print("\nWorking-tree changes:")
print(git_status if git_status else "(clean)")


# ---------------------------------------------------------------------
# train_model.py seed initialisation
# ---------------------------------------------------------------------

train_script = ROOT / "scripts" / "train_model.py"

if train_script.exists():
    text = train_script.read_text(
        encoding="utf-8",
        errors="replace"
    )

    init_seed_lines = [
        (i, line.strip())
        for i, line in enumerate(text.splitlines(), start=1)
        if re.search(r"\binit_seed\s*\(", line)
    ]

    print("\nExplicit init_seed calls found in train_model.py:")

    if init_seed_lines:
        for line_no, line in init_seed_lines:
            print(f"  Line {line_no}: {line}")
    else:
        print("  None found.")

    add_check(
        "3.2",
        "Final training runner contains explicit RecBole init_seed calls",
        len(init_seed_lines) >= 2,
        (
            f"Found {len(init_seed_lines)} explicit call(s). "
            "RecBole's standard pattern normally initialises before "
            "dataset preparation and again before model construction."
        ),
        severity="WARN",
    )
else:
    add_check(
        "3.2",
        "scripts/train_model.py exists",
        False,
        str(train_script),
        severity="WARN",
    )


show_checks("3.2")

Current repository HEAD : 8364b4f361c7d23dd39ba1ca7bd423dd288c86cd
Recorded result commit  : ['345ecc4f8cbd507c663a4c4a449a9cfb809323f5']

Working-tree changes:
M notebooks/01_data_exploration.ipynb
 M notebooks/02_preprocessing_checks.ipynb
 D notebooks/03_results_analysis.ipynb
?? notebooks/03_results_verification.ipynb

Explicit init_seed calls found in train_model.py:
  Line 1133: init_seed(
  Line 1154: init_seed(


,section,check,status,details
0,3.2,Recorded result commit is unique,PASS,['345ecc4f8cbd507c663a4c4a449a9cfb809323f5']
1,3.2,Current HEAD equals experimental result commit,WARN,Current=8364b4f361c7d23dd39ba1ca7bd423dd288c86...
2,3.2,Final training runner contains explicit RecBol...,PASS,Found 2 explicit call(s). RecBole's standard p...


# 4. Metric Integrity

## 4.1 Metric sanity checks

Verify that all evaluation metrics are present, bounded correctly, monotonic across K, and mathematically consistent with single-target leave-one-out evaluation.

In [7]:
METRIC_COLS = []

for split in ["validation", "test"]:
    for metric in ["recall", "ndcg", "hit", "mrr"]:
        for k in [5, 10, 20]:
            METRIC_COLS.append(f"{split}_{metric}@{k}")

missing_metric_columns = [
    col for col in METRIC_COLS
    if col not in results.columns
]

add_check(
    "4.1",
    "All expected metric columns exist",
    len(missing_metric_columns) == 0,
    missing_metric_columns,
)

if not missing_metric_columns:

    # -------------------------------------------------------------
    # Missing/non-finite values
    # -------------------------------------------------------------

    metric_matrix = results[METRIC_COLS].apply(
        pd.to_numeric,
        errors="coerce"
    )

    add_check(
        "4.1",
        "No missing metric values",
        not metric_matrix.isna().any().any(),
        f"Missing values: {int(metric_matrix.isna().sum().sum())}",
    )

    add_check(
        "4.1",
        "All metric values are finite",
        np.isfinite(metric_matrix.to_numpy()).all(),
    )

    # -------------------------------------------------------------
    # Metric range
    # -------------------------------------------------------------

    add_check(
        "4.1",
        "All ranking metrics lie between 0 and 1",
        (
            (metric_matrix >= 0)
            & (metric_matrix <= 1)
        ).all().all(),
        (
            f"Minimum={metric_matrix.min().min():.6f}, "
            f"Maximum={metric_matrix.max().max():.6f}"
        ),
    )

    # -------------------------------------------------------------
    # Monotonicity across K
    # -------------------------------------------------------------

    for split in ["validation", "test"]:
        for metric in ["recall", "hit", "ndcg", "mrr"]:
            values = results[
                [
                    f"{split}_{metric}@5",
                    f"{split}_{metric}@10",
                    f"{split}_{metric}@20",
                ]
            ].to_numpy()

            monotonic = (
                (values[:, 0] <= values[:, 1] + 1e-12)
                & (values[:, 1] <= values[:, 2] + 1e-12)
            ).all()

            add_check(
                "4.1",
                f"{split} {metric.upper()} is monotonic from K=5→10→20",
                monotonic,
            )

    # -------------------------------------------------------------
    # Leave-one-out mathematical relationships
    # -------------------------------------------------------------

    for split in ["validation", "test"]:
        for k in [5, 10, 20]:

            recall = results[f"{split}_recall@{k}"]
            hit = results[f"{split}_hit@{k}"]
            ndcg = results[f"{split}_ndcg@{k}"]
            mrr = results[f"{split}_mrr@{k}"]

            add_check(
                "4.1",
                f"{split}@{k}: Recall equals Hit",
                np.allclose(
                    recall,
                    hit,
                    atol=1e-12,
                ),
            )

            add_check(
                "4.1",
                f"{split}@{k}: MRR ≤ NDCG ≤ Recall",
                (
                    (mrr <= ndcg + 1e-12)
                    & (ndcg <= recall + 1e-12)
                ).all(),
            )

    # -------------------------------------------------------------
    # Selected validation metric
    # -------------------------------------------------------------

    add_check(
        "4.1",
        "best_valid_score equals validation NDCG@10",
        np.allclose(
            results["best_valid_score"],
            results["validation_ndcg@10"],
            atol=1e-12,
        ),
    )


metric_ranges = pd.DataFrame({
    "metric": METRIC_COLS,
    "min": [results[c].min() for c in METRIC_COLS],
    "max": [results[c].max() for c in METRIC_COLS],
})

display(metric_ranges)

show_checks("4.1")

,metric,min,max
0,validation_recall@5,0.001989,0.156281
1,validation_recall@10,0.003646,0.246934
2,validation_recall@20,0.006463,0.354823
3,validation_ndcg@5,0.001194,0.101701
4,validation_ndcg@10,0.001720,0.130920
5,validation_ndcg@20,0.002423,0.158015
6,validation_hit@5,0.001989,0.156281
7,validation_hit@10,0.003646,0.246934
8,validation_hit@20,0.006463,0.354823
9,validation_mrr@5,0.000931,0.083825


,section,check,status,details
0,4.1,All expected metric columns exist,PASS,[]
1,4.1,No missing metric values,PASS,Missing values: 0
2,4.1,All metric values are finite,PASS,
3,4.1,All ranking metrics lie between 0 and 1,PASS,"Minimum=0.000414, Maximum=0.354823"
4,4.1,validation RECALL is monotonic from K=5→10→20,PASS,
5,4.1,validation HIT is monotonic from K=5→10→20,PASS,
6,4.1,validation NDCG is monotonic from K=5→10→20,PASS,
7,4.1,validation MRR is monotonic from K=5→10→20,PASS,
8,4.1,test RECALL is monotonic from K=5→10→20,PASS,
9,4.1,test HIT is monotonic from K=5→10→20,PASS,


## 4.2 Relative-drop verification

Recalculate NDCG@10, Recall@10, and MRR@10 degradation from each model and seed's own full-data baseline.

In [8]:
RELATIVE_METRICS = {
    "test_ndcg@10": "relative_drop_test_ndcg@10",
    "test_recall@10": "relative_drop_test_recall@10",
    "test_mrr@10": "relative_drop_test_mrr@10",
}

baseline_reference = (
    results[results["scenario"] == "baseline"]
    [
        [
            "dataset",
            "model",
            "model_seed",
            *RELATIVE_METRICS.keys(),
        ]
    ]
    .copy()
)

for metric, stored_column in RELATIVE_METRICS.items():

    reference = baseline_reference[
        ["dataset", "model", "model_seed", metric]
    ].rename(
        columns={metric: "_baseline_metric"}
    )

    comparison = results.merge(
        reference,
        on=["dataset", "model", "model_seed"],
        how="left",
    )

    expected_drop = (
        comparison["_baseline_metric"]
        - comparison[metric]
    ) / comparison["_baseline_metric"]

    max_error = np.max(
        np.abs(
            expected_drop
            - comparison[stored_column]
        )
    )

    add_check(
        "4.2",
        f"{stored_column} recalculates exactly",
        np.allclose(
            expected_drop,
            comparison[stored_column],
            atol=1e-12,
        ),
        f"Maximum absolute error: {max_error:.3e}",
    )


baseline_rows = results["scenario"] == "baseline"

for stored_column in RELATIVE_METRICS.values():
    add_check(
        "4.2",
        f"{stored_column} is zero at baseline",
        np.allclose(
            results.loc[baseline_rows, stored_column],
            0,
            atol=1e-12,
        ),
    )


relative_summary = (
    results.groupby(
        ["dataset", "scenario", "retention_level"],
        as_index=False
    )
    .agg(
        mean_ndcg_drop=("relative_drop_test_ndcg@10", "mean"),
        min_ndcg_drop=("relative_drop_test_ndcg@10", "min"),
        max_ndcg_drop=("relative_drop_test_ndcg@10", "max"),
    )
)

display(relative_summary)

show_checks("4.2")

,dataset,scenario,retention_level,mean_ndcg_drop,min_ndcg_drop,max_ndcg_drop
0,amazon,baseline,100,0.000000,0.000000,0.000000
1,amazon,early,10,0.680466,0.091357,0.918299
2,amazon,early,25,0.627203,0.211996,0.894058
3,amazon,early,50,0.399829,0.204799,0.729076
4,amazon,global,10,0.630845,0.031744,0.906781
5,amazon,global,25,0.571507,0.054814,0.931022
6,amazon,global,50,0.339758,0.022639,0.733483
7,amazon,recent,10,0.591351,-0.000492,0.870078
8,amazon,recent,25,0.457891,0.002092,0.786870
9,amazon,recent,50,0.216214,-0.007444,0.529791


,section,check,status,details
0,4.2,relative_drop_test_ndcg@10 recalculates exactly,PASS,Maximum absolute error: 1.110e-16
1,4.2,relative_drop_test_recall@10 recalculates exactly,PASS,Maximum absolute error: 1.110e-16
2,4.2,relative_drop_test_mrr@10 recalculates exactly,PASS,Maximum absolute error: 1.110e-16
3,4.2,relative_drop_test_ndcg@10 is zero at baseline,PASS,
4,4.2,relative_drop_test_recall@10 is zero at baseline,PASS,
5,4.2,relative_drop_test_mrr@10 is zero at baseline,PASS,


# 5. RecBole Dataset Verification

## 5.1 Independent reconstruction

Read the actual `.inter` and `.item` files and independently verify the baseline statistics, fixed catalogue, leave-one-out targets, per-user retention counts, recent/early selection rules, and nested sparsity property.

In [9]:
DATA_FILE_REPORT = []

for dataset in ["movielens", "amazon"]:

    print("=" * 80)
    print(f"VERIFYING DATASET: {dataset.upper()}")
    print("=" * 80)

    dataset_results = results[
        results["dataset"] == dataset
    ].copy()

    expected_stats = EXPECTED_DATASET_STATS[dataset]

    # -----------------------------------------------------------------
    # Locate baseline files
    # -----------------------------------------------------------------

    baseline_row = dataset_results[
        dataset_results["scenario"] == "baseline"
    ].iloc[0]

    baseline_dir = ROOT / baseline_row["dataset_directory"]

    baseline_inter_path = find_single_file(
        baseline_dir,
        ".inter"
    )

    baseline_item_path = find_single_file(
        baseline_dir,
        ".item"
    )

    add_check(
        "5.1",
        f"{dataset}: baseline .inter exists",
        baseline_inter_path is not None,
        baseline_dir,
    )

    add_check(
        "5.1",
        f"{dataset}: baseline .item exists",
        baseline_item_path is not None,
        baseline_dir,
    )

    if baseline_inter_path is None or baseline_item_path is None:
        print(
            f"Cannot continue file-level verification for {dataset}."
        )
        continue

    # -----------------------------------------------------------------
    # Recalculate baseline hashes
    # -----------------------------------------------------------------

    actual_inter_hash = sha256_file(
        baseline_inter_path
    )

    actual_item_hash = sha256_file(
        baseline_item_path
    )

    add_check(
        "5.1",
        f"{dataset}: baseline .inter SHA256 matches result provenance",
        actual_inter_hash
        == baseline_row["dataset_inter_sha256"],
        (
            f"Actual={actual_inter_hash}, "
            f"Recorded={baseline_row['dataset_inter_sha256']}"
        ),
    )

    add_check(
        "5.1",
        f"{dataset}: baseline .item SHA256 matches result provenance",
        actual_item_hash
        == baseline_row["dataset_item_sha256"],
        (
            f"Actual={actual_item_hash}, "
            f"Recorded={baseline_row['dataset_item_sha256']}"
        ),
    )

    # -----------------------------------------------------------------
    # Read baseline files
    # -----------------------------------------------------------------

    baseline_inter = read_recbole_table(
        baseline_inter_path
    )

    baseline_items = read_recbole_table(
        baseline_item_path
    )

    required_inter_cols = {
        "user_id",
        "item_id",
        "sequence_order",
    }

    add_check(
        "5.1",
        f"{dataset}: baseline .inter contains required columns",
        required_inter_cols.issubset(
            baseline_inter.columns
        ),
        baseline_inter.columns.tolist(),
    )

    if not required_inter_cols.issubset(
        baseline_inter.columns
    ):
        continue

    # -----------------------------------------------------------------
    # Baseline basic integrity
    # -----------------------------------------------------------------

    n_users = baseline_inter["user_id"].nunique()
    n_items = baseline_inter["item_id"].nunique()
    n_interactions = len(baseline_inter)

    add_check(
        "5.1",
        f"{dataset}: expected baseline user count",
        n_users == expected_stats["users"],
        f"Expected={expected_stats['users']}, actual={n_users}",
    )

    add_check(
        "5.1",
        f"{dataset}: expected baseline item count",
        n_items == expected_stats["items"],
        f"Expected={expected_stats['items']}, actual={n_items}",
    )

    add_check(
        "5.1",
        f"{dataset}: expected baseline interaction count",
        n_interactions == expected_stats["interactions"],
        (
            f"Expected={expected_stats['interactions']}, "
            f"actual={n_interactions}"
        ),
    )

    add_check(
        "5.1",
        f"{dataset}: no duplicate user-item interactions",
        not baseline_inter.duplicated(
            ["user_id", "item_id"]
        ).any(),
        (
            f"Duplicates="
            f"{baseline_inter.duplicated(['user_id', 'item_id']).sum()}"
        ),
    )

    add_check(
        "5.1",
        f"{dataset}: sequence_order is unique within each user",
        not baseline_inter.duplicated(
            ["user_id", "sequence_order"]
        ).any(),
    )

    # -----------------------------------------------------------------
    # Catalogue coverage
    # -----------------------------------------------------------------

    if "item_id" not in baseline_items.columns:
        add_check(
            "5.1",
            f"{dataset}: .item contains item_id",
            False,
            baseline_items.columns.tolist(),
        )
        continue

    catalogue_set = set(
        baseline_items["item_id"].astype(str)
    )

    baseline_inter_item_set = set(
        baseline_inter["item_id"].astype(str)
    )

    add_check(
        "5.1",
        f"{dataset}: baseline interactions covered by item catalogue",
        baseline_inter_item_set.issubset(
            catalogue_set
        ),
        (
            f"Catalogue items={len(catalogue_set)}, "
            f"interaction items={len(baseline_inter_item_set)}"
        ),
    )

    add_check(
        "5.1",
        f"{dataset}: item catalogue has expected number of items",
        len(catalogue_set) == expected_stats["items"],
        (
            f"Expected={expected_stats['items']}, "
            f"actual={len(catalogue_set)}"
        ),
    )

    # -----------------------------------------------------------------
    # Independently reconstruct baseline leave-one-out split
    # -----------------------------------------------------------------

    baseline_train, baseline_valid, baseline_test = (
        split_leave_one_out(
            baseline_inter
        )
    )

    add_check(
        "5.1",
        f"{dataset}: expected baseline training interactions",
        len(baseline_train) == expected_stats["train"],
        (
            f"Expected={expected_stats['train']}, "
            f"actual={len(baseline_train)}"
        ),
    )

    add_check(
        "5.1",
        f"{dataset}: exactly one validation target per user",
        len(baseline_valid)
        == expected_stats["validation"]
        == n_users,
        f"Validation rows={len(baseline_valid)}",
    )

    add_check(
        "5.1",
        f"{dataset}: exactly one test target per user",
        len(baseline_test)
        == expected_stats["test"]
        == n_users,
        f"Test rows={len(baseline_test)}",
    )

    baseline_train_counts = (
        baseline_train.groupby("user_id")
        .size()
        .sort_index()
    )

    add_check(
        "5.1",
        f"{dataset}: every user has at least one baseline training interaction",
        (baseline_train_counts >= 1).all(),
        (
            f"Minimum baseline train history="
            f"{baseline_train_counts.min()}"
        ),
    )

    # Key columns used when comparing exact retained rows.
    exact_cols = [
        "user_id",
        "item_id",
        "sequence_order",
    ]

    if "timestamp" in baseline_inter.columns:
        exact_cols.append("timestamp")

    # Store random/global selections for nesting check.
    global_hashes = {}

    # -----------------------------------------------------------------
    # Verify every sparse condition
    # -----------------------------------------------------------------

    sparse_conditions = (
        dataset_results[
            dataset_results["scenario"] != "baseline"
        ]
        [
            [
                "scenario",
                "retention_level",
                "dataset_directory",
                "dataset_inter_sha256",
                "dataset_item_sha256",
                "actual_training_retention_percent",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            ["scenario", "retention_level"],
            ascending=[True, False]
        )
    )

    for _, condition in sparse_conditions.iterrows():

        scenario = condition["scenario"]
        retention = int(condition["retention_level"])

        condition_dir = (
            ROOT / condition["dataset_directory"]
        )

        inter_path = find_single_file(
            condition_dir,
            ".inter"
        )

        item_path = find_single_file(
            condition_dir,
            ".item"
        )

        label = (
            f"{dataset}/{scenario}/{retention}"
        )

        add_check(
            "5.1",
            f"{label}: .inter exists",
            inter_path is not None,
            condition_dir,
        )

        add_check(
            "5.1",
            f"{label}: .item exists",
            item_path is not None,
            condition_dir,
        )

        if inter_path is None or item_path is None:
            continue

        # -------------------------------------------------------------
        # File hashes
        # -------------------------------------------------------------

        condition_inter_hash = sha256_file(
            inter_path
        )

        condition_item_hash = sha256_file(
            item_path
        )

        add_check(
            "5.1",
            f"{label}: .inter hash matches recorded hash",
            condition_inter_hash
            == condition["dataset_inter_sha256"],
        )

        add_check(
            "5.1",
            f"{label}: fixed .item file is byte-identical to baseline",
            condition_item_hash
            == actual_item_hash
            == condition["dataset_item_sha256"],
        )

        # -------------------------------------------------------------
        # Load sparse data
        # -------------------------------------------------------------

        sparse_inter = read_recbole_table(
            inter_path
        )

        sparse_users = set(
            sparse_inter["user_id"].astype(str)
        )

        baseline_users = set(
            baseline_inter["user_id"].astype(str)
        )

        add_check(
            "5.1",
            f"{label}: all baseline users remain",
            sparse_users == baseline_users,
            (
                f"Baseline users={len(baseline_users)}, "
                f"sparse users={len(sparse_users)}"
            ),
        )

        add_check(
            "5.1",
            f"{label}: no duplicate user-item interactions",
            not sparse_inter.duplicated(
                ["user_id", "item_id"]
            ).any(),
        )

        add_check(
            "5.1",
            f"{label}: sequence_order remains unique within users",
            not sparse_inter.duplicated(
                ["user_id", "sequence_order"]
            ).any(),
        )

        sparse_item_set = set(
            sparse_inter["item_id"].astype(str)
        )

        add_check(
            "5.1",
            f"{label}: every interaction item is in fixed catalogue",
            sparse_item_set.issubset(
                catalogue_set
            ),
        )

        # -------------------------------------------------------------
        # Reconstruct leave-one-out
        # -------------------------------------------------------------

        sparse_train, sparse_valid, sparse_test = (
            split_leave_one_out(
                sparse_inter
            )
        )

        # Validation/test must remain exactly unchanged.
        add_check(
            "5.1",
            f"{label}: validation targets exactly match baseline",
            same_rows(
                sparse_valid,
                baseline_valid,
                exact_cols,
            ),
        )

        add_check(
            "5.1",
            f"{label}: test targets exactly match baseline",
            same_rows(
                sparse_test,
                baseline_test,
                exact_cols,
            ),
        )

        # Sparse training data must only contain baseline training rows.
        add_check(
            "5.1",
            f"{label}: sparse training is a subset of baseline training",
            subset_rows(
                sparse_train,
                baseline_train,
                exact_cols,
            ),
        )

        # -------------------------------------------------------------
        # Per-user ceil retention rule
        # -------------------------------------------------------------

        fraction = retention / 100.0

        expected_counts = np.ceil(
            baseline_train_counts * fraction
        ).astype(int)

        expected_counts = expected_counts.clip(
            lower=1
        )

        sparse_counts = (
            sparse_train.groupby("user_id")
            .size()
            .reindex(
                baseline_train_counts.index,
                fill_value=0,
            )
            .astype(int)
        )

        counts_correct = (
            sparse_counts.to_numpy()
            == expected_counts.to_numpy()
        ).all()

        add_check(
            "5.1",
            (
                f"{label}: per-user retained count equals "
                f"max(1, ceil(n × {fraction:g}))"
            ),
            counts_correct,
            (
                f"Mismatched users="
                f"{int((sparse_counts != expected_counts).sum())}"
            ),
        )

        # -------------------------------------------------------------
        # Actual aggregate retention
        # -------------------------------------------------------------

        recalculated_retention = (
            len(sparse_train)
            / len(baseline_train)
            * 100
        )

        recorded_retention = float(
            condition[
                "actual_training_retention_percent"
            ]
        )

        add_check(
            "5.1",
            f"{label}: recorded actual retention is correct",
            np.isclose(
                recalculated_retention,
                recorded_retention,
                atol=1e-9,
            ),
            (
                f"Recalculated={recalculated_retention:.9f}%, "
                f"recorded={recorded_retention:.9f}%"
            ),
        )

        # -------------------------------------------------------------
        # Exact recent / early selection
        # -------------------------------------------------------------

        if scenario in {"recent", "early"}:

            ordered_train = baseline_train.sort_values(
                ["user_id", "sequence_order"],
                kind="mergesort",
            ).copy()

            ordered_train["_pos"] = (
                ordered_train.groupby(
                    "user_id",
                    sort=False
                ).cumcount()
            )

            ordered_train["_n"] = (
                ordered_train.groupby(
                    "user_id",
                    sort=False
                )["user_id"]
                .transform("size")
            )

            ordered_train["_k"] = np.ceil(
                ordered_train["_n"] * fraction
            ).astype(int)

            ordered_train["_k"] = (
                ordered_train["_k"]
                .clip(lower=1)
            )

            if scenario == "early":
                mask = (
                    ordered_train["_pos"]
                    < ordered_train["_k"]
                )

            else:  # recent
                mask = (
                    ordered_train["_pos"]
                    >= (
                        ordered_train["_n"]
                        - ordered_train["_k"]
                    )
                )

            expected_train = ordered_train[
                mask
            ]

            add_check(
                "5.1",
                f"{label}: exact {scenario} interactions retained",
                same_rows(
                    sparse_train,
                    expected_train,
                    exact_cols,
                ),
            )

        # -------------------------------------------------------------
        # Save global/random rows for nesting test
        # -------------------------------------------------------------

        if scenario == "global":
            global_hashes[retention] = np.unique(
                row_hashes(
                    sparse_train,
                    exact_cols,
                )
            )

        DATA_FILE_REPORT.append({
            "dataset": dataset,
            "scenario": scenario,
            "retention_level": retention,
            "users": sparse_inter["user_id"].nunique(),
            "total_interactions": len(sparse_inter),
            "training_interactions": len(sparse_train),
            "validation_interactions": len(sparse_valid),
            "test_interactions": len(sparse_test),
            "unique_interaction_items": sparse_inter["item_id"].nunique(),
            "catalogue_items": len(catalogue_set),
            "actual_retention_percent": recalculated_retention,
        })

        del sparse_inter
        del sparse_train
        del sparse_valid
        del sparse_test

    # -----------------------------------------------------------------
    # Verify nested random-per-user subsets
    # -----------------------------------------------------------------

    if all(level in global_hashes for level in [10, 25, 50]):

        nested_10_25 = (
            len(
                np.setdiff1d(
                    global_hashes[10],
                    global_hashes[25],
                    assume_unique=True,
                )
            )
            == 0
        )

        nested_25_50 = (
            len(
                np.setdiff1d(
                    global_hashes[25],
                    global_hashes[50],
                    assume_unique=True,
                )
            )
            == 0
        )

        add_check(
            "5.1",
            f"{dataset}: random 10% is a subset of random 25%",
            nested_10_25,
        )

        add_check(
            "5.1",
            f"{dataset}: random 25% is a subset of random 50%",
            nested_25_50,
        )

    print(f"Completed {dataset}.")


# ---------------------------------------------------------------------
# Display independent dataset summary
# ---------------------------------------------------------------------

data_file_report = pd.DataFrame(
    DATA_FILE_REPORT
)

if not data_file_report.empty:
    display(
        data_file_report.sort_values(
            ["dataset", "scenario", "retention_level"]
        ).reset_index(drop=True)
    )

show_checks("5.1")

VERIFYING DATASET: MOVIELENS
Completed movielens.
VERIFYING DATASET: AMAZON
Completed amazon.


,dataset,scenario,retention_level,users,total_interactions,training_interactions,validation_interactions,test_interactions,unique_interaction_items,catalogue_items,actual_retention_percent
0,amazon,early,10,63413,201582,74756,63413,63413,18545,19022,18.380032
1,amazon,early,25,63413,247894,121068,63413,63413,18837,19022,29.766623
2,amazon,early,50,63413,349351,222525,63413,63413,18997,19022,54.711549
3,amazon,global,10,63413,201582,74756,63413,63413,18424,19022,18.380032
4,amazon,global,25,63413,247894,121068,63413,63413,18811,19022,29.766623
5,amazon,global,50,63413,349351,222525,63413,63413,19004,19022,54.711549
6,amazon,recent,10,63413,201582,74756,63413,63413,18190,19022,18.380032
7,amazon,recent,25,63413,247894,121068,63413,63413,18627,19022,29.766623
8,amazon,recent,50,63413,349351,222525,63413,63413,18971,19022,54.711549
9,movielens,early,10,6034,70990,58922,6034,6034,2860,3125,10.478599


,section,check,status,details
0,5.1,movielens: baseline .inter exists,PASS,C:\Research Project\recommender-sparsity-honou...
1,5.1,movielens: baseline .item exists,PASS,C:\Research Project\recommender-sparsity-honou...
2,5.1,movielens: baseline .inter SHA256 matches resu...,PASS,Actual=a442c7e08a460016c8c096322fd4e608b26cbc9...
3,5.1,movielens: baseline .item SHA256 matches resul...,PASS,Actual=f135ebc73d0022e1e454368d2f6256418caa23d...
4,5.1,movielens: baseline .inter contains required c...,PASS,"['user_id', 'item_id', 'timestamp', 'sequence_..."
...,...,...,...,...
277,5.1,amazon/recent/10: per-user retained count equa...,PASS,Mismatched users=0
278,5.1,amazon/recent/10: recorded actual retention is...,PASS,"Recalculated=18.380031668%, recorded=18.380032..."
279,5.1,amazon/recent/10: exact recent interactions re...,PASS,
280,5.1,amazon: random 10% is a subset of random 25%,PASS,


# 6. Raw-to-Processed Traceability

## 6.1 Final result provenance

Verify that the 480 processed experiment rows correspond to final runs recorded in the raw experiment log.

In [10]:
raw_path = (
    ROOT
    / "results"
    / "raw"
    / "experiment_results.csv"
)

if raw_path.exists():

    raw = pd.read_csv(raw_path)

    add_check(
        "6.1",
        "Raw experiment_results.csv exists",
        True,
        raw_path,
    )

    if "run_type" not in raw.columns:
        add_check(
            "6.1",
            "Raw results contain run_type",
            False,
        )
    else:
        raw_final = raw[
            raw["run_type"] == "final"
        ].copy()

        raw_final["model_seed"] = pd.to_numeric(
            raw_final["model_seed"],
            errors="coerce"
        ).astype("Int64")

        raw_final["retention_level"] = pd.to_numeric(
            raw_final["retention_level"],
            errors="coerce"
        ).astype("Int64")

        print(f"Raw rows total : {len(raw)}")
        print(f"Raw final rows : {len(raw_final)}")
        print(f"Processed rows : {len(results)}")

        # -------------------------------------------------------------
        # Experiment-key comparison
        # -------------------------------------------------------------

        raw_keys = raw_final[
            KEY_COLS
        ].drop_duplicates()

        processed_keys = results[
            KEY_COLS
        ].drop_duplicates()

        key_comparison = processed_keys.merge(
            raw_keys,
            on=KEY_COLS,
            how="outer",
            indicator=True,
        )

        processed_missing_from_raw = (
            key_comparison[
                key_comparison["_merge"]
                == "left_only"
            ]
        )

        raw_missing_from_processed = (
            key_comparison[
                key_comparison["_merge"]
                == "right_only"
            ]
        )

        add_check(
            "6.1",
            "All processed experiment keys exist in raw final results",
            processed_missing_from_raw.empty,
            (
                f"Missing="
                f"{len(processed_missing_from_raw)}"
            ),
        )

        add_check(
            "6.1",
            "No unique raw final experiment is omitted from processed results",
            raw_missing_from_processed.empty,
            (
                f"Unique raw-only combinations="
                f"{len(raw_missing_from_processed)}"
            ),
        )

        add_check(
            "6.1",
            "Raw final results contain exactly 480 unique experiment keys",
            len(raw_keys) == 480,
            (
                f"Raw final rows={len(raw_final)}, "
                f"unique keys={len(raw_keys)}"
            ),
        )

        # -------------------------------------------------------------
        # Detect duplicate final runs in raw file
        # -------------------------------------------------------------

        raw_duplicate_keys = (
            raw_final[
                raw_final.duplicated(
                    KEY_COLS,
                    keep=False
                )
            ]
            .sort_values(KEY_COLS)
        )

        duplicate_excess = (
            len(raw_final)
            - len(raw_keys)
        )

        print(
            f"Duplicate/excess final raw rows: "
            f"{duplicate_excess}"
        )

        # One historical duplicate is already known from processing.
        add_check(
            "6.1",
            "Raw duplicate final rows are accounted for",
            duplicate_excess >= 0,
            (
                f"Raw final={len(raw_final)}, "
                f"unique final keys={len(raw_keys)}, "
                f"excess={duplicate_excess}"
            ),
        )

        if not raw_duplicate_keys.empty:
            print("\nRaw duplicate experiment keys:")
            display(
                raw_duplicate_keys[
                    KEY_COLS
                    + [
                        "timestamp",
                        "test_ndcg@10",
                    ]
                ]
            )

        # -------------------------------------------------------------
        # Stronger provenance fingerprint comparison
        # -------------------------------------------------------------

        fingerprint_candidates = [
            "dataset",
            "scenario",
            "retention_level",
            "model",
            "model_seed",
            "timestamp",
            "config_sha256",
            "dataset_inter_sha256",
            "dataset_item_sha256",
            "best_valid_score",
            "test_ndcg@10",
            "test_recall@10",
            "test_mrr@10",
        ]

        fingerprint_cols = [
            c for c in fingerprint_candidates
            if c in raw_final.columns
            and c in results.columns
        ]

        def make_fingerprints(df, cols):
            temp = df[cols].copy()

            for col in temp.select_dtypes(
                include=[np.number]
            ).columns:
                temp[col] = temp[col].round(12)

            temp = (
                temp
                .fillna("<NA>")
                .astype(str)
            )

            return set(
                pd.util.hash_pandas_object(
                    temp,
                    index=False
                ).to_numpy(dtype=np.uint64)
            )

        raw_fingerprints = make_fingerprints(
            raw_final,
            fingerprint_cols,
        )

        processed_fingerprints = make_fingerprints(
            results,
            fingerprint_cols,
        )

        add_check(
            "6.1",
            "Every processed result fingerprint exists in raw final results",
            processed_fingerprints.issubset(
                raw_fingerprints
            ),
            (
                f"Processed fingerprints="
                f"{len(processed_fingerprints)}, "
                f"raw={len(raw_fingerprints)}"
            ),
        )

else:
    add_check(
        "6.1",
        "Raw experiment_results.csv exists",
        False,
        raw_path,
    )


show_checks("6.1")

Raw rows total : 502
Raw final rows : 481
Processed rows : 480
Duplicate/excess final raw rows: 1

Raw duplicate experiment keys:


,dataset,scenario,retention_level,model,model_seed,timestamp,test_ndcg@10
40,movielens,baseline,100,BERT4Rec,2026,2026-08-23T09:33:24,0.104292
45,movielens,baseline,100,BERT4Rec,2026,2026-08-23T11:59:30,0.109688


,section,check,status,details
0,6.1,Raw experiment_results.csv exists,PASS,C:\Research Project\recommender-sparsity-honou...
1,6.1,All processed experiment keys exist in raw fin...,PASS,Missing=0
2,6.1,No unique raw final experiment is omitted from...,PASS,Unique raw-only combinations=0
3,6.1,Raw final results contain exactly 480 unique e...,PASS,"Raw final rows=481, unique keys=480"
4,6.1,Raw duplicate final rows are accounted for,PASS,"Raw final=481, unique final keys=480, excess=1"
5,6.1,Every processed result fingerprint exists in r...,PASS,"Processed fingerprints=480, raw=481"


# 7. Hyperparameter Tuning Verification

## 7.1 Baseline tuning coverage

Check that all tunable models have recorded baseline-selected configurations and identify any documented exceptions.

In [11]:
best_config_candidates = list(
    (ROOT / "results" / "tuning").rglob(
        "best_configs.csv"
    )
)

if not best_config_candidates:

    add_check(
        "7.1",
        "best_configs.csv exists",
        False,
        "No best_configs.csv found.",
        severity="WARN",
    )

else:
    best_configs_path = best_config_candidates[0]
    best_configs = pd.read_csv(
        best_configs_path
    )

    print(f"Best configs file: {best_configs_path}")
    display(best_configs)

    # Try to identify dataset/model columns robustly.
    lower_map = {
        c.lower(): c
        for c in best_configs.columns
    }

    dataset_col = lower_map.get("dataset")
    model_col = lower_map.get("model")

    if dataset_col is None or model_col is None:

        add_check(
            "7.1",
            "Tuning table contains dataset and model columns",
            False,
            best_configs.columns.tolist(),
            severity="WARN",
        )

    else:

        tuning_pairs = set(
            zip(
                best_configs[
                    dataset_col
                ].astype(str).str.lower(),
                best_configs[
                    model_col
                ].astype(str),
            )
        )

        tunable_models = set(MODELS) - {"Pop"}

        expected_tuning_pairs = {
            (dataset, model)
            for dataset in [
                "movielens",
                "amazon",
            ]
            for model in tunable_models
        }

        missing_tuning = (
            expected_tuning_pairs
            - tuning_pairs
        )

        known_exception = {
            ("amazon", "LightGCN")
        }

        unexpected_missing = (
            missing_tuning
            - known_exception
        )

        add_check(
            "7.1",
            "All tuning selections except documented Amazon LightGCN exception exist",
            len(unexpected_missing) == 0,
            (
                f"Missing tuning pairs: "
                f"{sorted(missing_tuning)}"
            ),
        )

        add_check(
            "7.1",
            "Amazon LightGCN has a completed tuning selection",
            (
                "amazon",
                "LightGCN"
            ) in tuning_pairs,
            (
                "If this remains WARN, explicitly document "
                "Amazon LightGCN as the tuning exception."
            ),
            severity="WARN",
        )


show_checks("7.1")

Best configs file: C:\Research Project\recommender-sparsity-honours\results\tuning\best_configs.csv


,dataset,model,candidate_index,seed,validation_ndcg@10,training_time_seconds,parameter_overrides,parameter_overrides_json,source_file,param_k,param_embedding_size,param_weight_decay,param_reg_weight,param_mf_embedding_size,param_mlp_embedding_size,param_latent_dimension,param_dropout_prob,param_hidden_size,param_n_layers,param_hidden_dropout_prob
0,movielens,ItemKNN,1,2025,0.040759,3.772547,{'k': 50},"{""k"": 50}",results\tuning\movielens\itemknn_tuning.csv,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,movielens,BPR,4,2025,0.045675,226.963633,"{'embedding_size': 64, 'weight_decay': '1e-6'}","{""embedding_size"": 64, ""weight_decay"": ""1e-6""}",results\tuning\movielens\bpr_tuning.csv,NaN,64.0,0.000001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,movielens,EASE,3,2025,0.054256,3.212770,{'reg_weight': 500.0},"{""reg_weight"": 500.0}",results\tuning\movielens\ease_tuning.csv,NaN,NaN,NaN,500.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,movielens,NeuMF,3,2025,0.040354,150.233343,"{'mf_embedding_size': 64, 'mlp_embedding_size'...","{""mf_embedding_size"": 64, ""mlp_embedding_size""...",results\tuning\movielens\neumf_tuning.csv,NaN,NaN,NaN,NaN,64.0,32.0,NaN,NaN,NaN,NaN,NaN
4,movielens,MultiVAE,4,2025,0.049842,181.026404,"{'latent_dimension': 200, 'dropout_prob': 0.5}","{""dropout_prob"": 0.5, ""latent_dimension"": 200}",results\tuning\movielens\multivae_tuning.csv,NaN,NaN,NaN,NaN,NaN,NaN,200.0,0.5,NaN,NaN,NaN
5,movielens,GRU4Rec,4,2025,0.127791,442.399111,"{'hidden_size': 128, 'dropout_prob': 0.2}","{""dropout_prob"": 0.2, ""hidden_size"": 128}",results\tuning\movielens\gru4rec_tuning.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.2,128.0,NaN,NaN
6,movielens,SASRec,3,2025,0.127990,1354.185274,"{'n_layers': 2, 'hidden_dropout_prob': 0.1}","{""hidden_dropout_prob"": 0.1, ""n_layers"": 2}",results\tuning\movielens\sasrec_tuning.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.1
7,movielens,BERT4Rec,3,2025,0.111891,1947.525922,"{'n_layers': 2, 'hidden_dropout_prob': 0.1}","{""hidden_dropout_prob"": 0.1, ""n_layers"": 2}",results\tuning\movielens\bert4rec_tuning.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.1
8,movielens,LightGCN,1,2025,0.044614,904.110585,"{'n_layers': 1, 'reg_weight': '1e-5'}","{""n_layers"": 1, ""reg_weight"": ""1e-5""}",results\tuning\movielens\lightgcn_tuning.csv,NaN,NaN,NaN,0.00001,NaN,NaN,NaN,NaN,NaN,1.0,NaN
9,amazon,ItemKNN,3,2025,0.041242,35.321761,{'k': 200},"{""k"": 200}",results\tuning\amazon\itemknn_tuning.csv,200.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,section,check,status,details
0,7.1,All tuning selections except documented Amazon...,PASS,"Missing tuning pairs: [('amazon', 'LightGCN')]"
1,7.1,Amazon LightGCN has a completed tuning selection,WARN,"If this remains WARN, explicitly document Amaz..."


# 8. Automated Project Tests

## 8.1 Test suite

Run the repository tests, including sparsity invariants, item-catalogue checks, and the BERT4Rec patch tests.

In [12]:
pytest_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
    ],
    cwd=ROOT,
    capture_output=True,
    text=True,
)

print(pytest_result.stdout)

if pytest_result.stderr.strip():
    print("\nSTDERR:")
    print(pytest_result.stderr)

add_check(
    "8.1",
    "Full pytest suite passes",
    pytest_result.returncode == 0,
    (
        f"Return code="
        f"{pytest_result.returncode}"
    ),
)

show_checks("8.1")

..........                                                               [100%]
10 passed in 4.18s



,section,check,status,details
0,8.1,Full pytest suite passes,PASS,Return code=0


# 9. Final Verification Summary

## 9.1 Final status

Combine all verification checks, export the verification report, and determine whether the final experiment set is ready for analysis.

In [13]:
checks_df = pd.DataFrame(CHECKS)

status_order = {
    "FAIL": 0,
    "WARN": 1,
    "PASS": 2,
}

checks_df["_order"] = (
    checks_df["status"]
    .map(status_order)
)

checks_df = (
    checks_df
    .sort_values(
        ["_order", "section", "check"]
    )
    .drop(columns="_order")
    .reset_index(drop=True)
)

display(checks_df)


# ---------------------------------------------------------------------
# Summary counts
# ---------------------------------------------------------------------

status_counts = (
    checks_df["status"]
    .value_counts()
    .reindex(
        ["PASS", "WARN", "FAIL"],
        fill_value=0,
    )
)

print("\nVerification totals")
print("-------------------")
print(f"PASS : {status_counts['PASS']}")
print(f"WARN : {status_counts['WARN']}")
print(f"FAIL : {status_counts['FAIL']}")


# ---------------------------------------------------------------------
# Export verification artifacts
# ---------------------------------------------------------------------

checks_path = (
    VERIFICATION_DIR
    / "verification_checks.csv"
)

checks_df.to_csv(
    checks_path,
    index=False,
)

condition_summary.to_csv(
    VERIFICATION_DIR
    / "condition_inventory.csv",
    index=False,
)

inventory.to_csv(
    VERIFICATION_DIR
    / "experiment_inventory.csv",
    index=False,
)

if "data_file_report" in globals() and not data_file_report.empty:
    data_file_report.to_csv(
        VERIFICATION_DIR
        / "dataset_integrity_summary.csv",
        index=False,
    )


summary = {
    "pass": int(status_counts["PASS"]),
    "warn": int(status_counts["WARN"]),
    "fail": int(status_counts["FAIL"]),
    "total_checks": int(len(checks_df)),
    "datasets": sorted(
        results["dataset"].unique().tolist()
    ),
    "processed_rows": int(len(results)),
    "result_git_commits": (
        results["git_commit"]
        .dropna()
        .unique()
        .tolist()
    ),
}

with open(
    VERIFICATION_DIR
    / "verification_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )


print(
    f"\nVerification reports saved to:\n"
    f"{VERIFICATION_DIR}"
)


# ---------------------------------------------------------------------
# Final decision
# ---------------------------------------------------------------------

if status_counts["FAIL"] > 0:

    print("\n" + "=" * 70)
    print("FINAL RESULT VERIFICATION: FAILED")
    print("=" * 70)
    print(
        "Resolve all FAIL checks before beginning "
        "the formal results analysis."
    )

elif status_counts["WARN"] > 0:

    print("\n" + "=" * 70)
    print("FINAL RESULT VERIFICATION: PASSED WITH WARNINGS")
    print("=" * 70)
    print(
        "The result set passed all critical checks, "
        "but the WARN items should be reviewed and documented."
    )

else:

    print("\n" + "=" * 70)
    print("FINAL RESULT VERIFICATION: PASSED")
    print("=" * 70)
    print(
        "The final experiment set is verified and "
        "ready for formal analysis."
    )

,section,check,status,details
0,3.2,Current HEAD equals experimental result commit,WARN,Current=8364b4f361c7d23dd39ba1ca7bd423dd288c86...
1,7.1,Amazon LightGCN has a completed tuning selection,WARN,"If this remains WARN, explicitly document Amaz..."
2,1.2,Both expected datasets loaded,PASS,"Found: ['amazon', 'movielens']"
3,1.2,Result tables contain 480 rows in total,PASS,Rows found: 480
4,2.1,Exactly 10 expected models,PASS,"['BERT4Rec', 'BPR', 'EASE', 'GRU4Rec', 'ItemKN..."
...,...,...,...,...
359,6.1,Raw duplicate final rows are accounted for,PASS,"Raw final=481, unique final keys=480, excess=1"
360,6.1,Raw experiment_results.csv exists,PASS,C:\Research Project\recommender-sparsity-honou...
361,6.1,Raw final results contain exactly 480 unique e...,PASS,"Raw final rows=481, unique keys=480"
362,7.1,All tuning selections except documented Amazon...,PASS,"Missing tuning pairs: [('amazon', 'LightGCN')]"



Verification totals
-------------------
PASS : 362
WARN : 2
FAIL : 0

Verification reports saved to:
C:\Research Project\recommender-sparsity-honours\results\verification

FINAL RESULT VERIFICATION: PASSED WITH WARNINGS
The result set passed all critical checks, but the WARN items should be reviewed and documented.
